# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring a Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

Explore and analyze FAIR^2-compliant mass spectrometry data.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Print out main metadata attributes
md = dataset.metadata
print(f"Dataset name:", md.name)
print(f"Dataset description:\n{md.description}\n")
print(f"Dataset @id: {md.id}")
print(f"Dataset version: {md.version}")
print(f"Dataset license: {md.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All references use `@id`s for reliability and reproducibility.

In [ ]:
# List all record sets, their @id, and the fields/columns they contain
print("Available Record Sets:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"\n- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    # List fields
    print("  Fields/Columns:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) [dataType: {field.data_type}]")

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis. Use `@id` values for all entities.

In [ ]:
# Collect all record set @id values to read them into DataFrames
recset_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
# Optionally print the record set ids we're working with:
print("Record set @ids:", recset_ids)

# Load all record sets into DataFrames
for record_set_id in recset_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id}, shape = {df.shape}")

# Choose the main record set (typically the largest or most relevant by name or description)
if recset_ids:
    main_record_set = recset_ids[0]
    print(f"\nColumns for {main_record_set}: {dataframes[main_record_set].columns.tolist()}")
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All field selections use `@id` references.

In [ ]:
# Choose a numeric field and a group field for demonstration
# For this dataset, common numeric fields might include normalized_abundance, coverage, and group fields like accession or description
# Let's try to pick them by looking up all available fields from the main_record_set
main_fields = dataset.metadata["record_sets"][0].fields

# Detect a likely numeric field (search for 'coverage', 'abundance', 'peptide_count', etc)
potential_numeric_fields = [f for f in main_fields if f.data_type in ('Float', 'Number', 'Integer')]
# Use the first one found
numeric_field = potential_numeric_fields[0].id if potential_numeric_fields else dataframes[main_record_set].select_dtypes('number').columns[0]
print(f"Using numeric field @id: {numeric_field}")

# Detect a potential group field
potential_group_fields = [f for f in main_fields if f.data_type=='Text']
group_field = potential_group_fields[0].id if potential_group_fields else dataframes[main_record_set].columns[0]
print(f"Using group (categorical) field @id: {group_field}")

df = dataframes[main_record_set]

# Filter, normalize, group
if numeric_field in df.columns:
    # Handle missing or non-numeric values
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered {len(filtered_df)} records with {numeric_field} > mean ({threshold:.4f})")
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    # Show head
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group by group_field if present
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print(f"No numeric field identified in {main_record_set}")

## 5. Visualization
Visualize data distributions or relationships between fields using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# Boxplot by group field
if group_field in df.columns and numeric_field in df.columns:
    # Limit to a few groups to keep plot readable
    group_counts = df[group_field].value_counts().nlargest(6).index.tolist()
    subset = df[df[group_field].isin(group_counts)]
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=subset)
    plt.title(f"{numeric_field} by {group_field} (Top 6 groups)")
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR^2 Mass Spectrometry dataset using the Croissant ecosystem. We explored the available record sets, fields referenced by `@id`, filtered records by quantitative criteria, normalized measurements, performed simple aggregations, and visualized the resulting distributions. Further domain-specific bioinformatics analyses can be layered on this reproducible foundation using the loaded DataFrames.